# Сверка ЧОД и Фин.Рез (Excel vs файлы из озера)

Этот ноутбук сравнивает значения `ЧОД`, `Фин.Рез` и `Количество терминалов` между тремя Excel-отчетами (январь, февраль, март) и тремя файлами из озера.

Что проверяется:
- агрегаты по месяцам;
- расхождения по ИНН;
- вклад разных типов причин (`only_excel`, `only_lake`, `value_mismatch`).

In [ ]:
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
# Параметры Excel (каждый файл соответствует своему месяцу)
excel_sources = [
    {
        'report_month': '2026-01-01',
        'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
        'header': 1,
    },
    {
        'report_month': '2026-02-01',
        'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
        'header': 0,
    },
    {
        'report_month': '2026-03-01',
        'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
        'header': 0,
    },
]

# Параметры файлов из озера (каждый файл соответствует своему месяцу)
lake_sources = [
    {
        'report_month': '2026-01-01',
        'path': '/home/jovyan/documents/Equaring/Проверки/Фин.скрипт/datamart_01_month_acquiring_2026_05.csv',
        'sep': ',',
    },
    {
        'report_month': '2026-02-01',
        'path': '/home/jovyan/documents/Equaring/Проверки/Фин.скрипт/datamart_02_month_acquiring_2026_05.csv',
        'sep': ',',
    },
    {
        'report_month': '2026-03-01',
        'path': '/home/jovyan/documents/Equaring/Проверки/Фин.скрипт/datamart_03_month_acquiring_2026_05.csv',
        'sep': ',',
    },
]

# Основные названия и алиасы колонок
excel_col_inn = 'ИНН'
excel_col_chod = 'ЧОД'
excel_col_finrez = 'Фин.Рез'
excel_col_term_cnt = 'Количество терминалов'

lake_col_inn = 'inn'
lake_col_chod = 'chod'
lake_col_finrez = 'fin_result'
lake_col_term_cnt = 'term_cnt'

excel_inn_aliases = ['ИНН', 'INN']
excel_chod_aliases = ['ЧОД', 'CHOD']
excel_finrez_aliases = ['Фин.Рез', 'Фин. Рез.', 'ФИН.РЕЗ', 'ФИНРЕЗ', 'ФИН РЕЗ', 'ФИН_РЕЗ', 'FIN_RESULT']
excel_term_cnt_aliases = ['Количество терминалов', 'Кол-во терминалов', 'Терминалы', 'term_cnt', 'TERM_CNT']

lake_inn_aliases = ['inn', 'ИНН']
lake_chod_aliases = ['chod', 'ЧОД']
lake_finrez_aliases = ['fin_result', 'finrez', 'Фин.Рез']
lake_term_cnt_aliases = ['term_cnt', 'Терминалы', 'Количество терминалов']

In [ ]:
def normalize_inn(value):
    if pd.isna(value):
        return None
    s = re.sub(r'\D+', '', str(value))
    return s if s else None


def normalize_colname(value):
    s = str(value).strip().lower().replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s)
    s = s.replace('.', '').replace(',', '').replace(';', '').replace(':', '')
    s = s.replace('-', '').replace('_', '')
    return s


def pick_column(columns, aliases, col_label, file_path):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]

    available = ', '.join([str(c) for c in columns])
    raise ValueError(
        f"В файле {file_path} не найдена колонка '{col_label}'. Доступные колонки: {available}"
    )


def parse_numeric(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s:
        return np.nan

    s = s.replace('\xa0', '').replace(' ', '')
    if ',' in s and '.' in s:
        # Если есть оба разделителя, считаем, что точка - тысячные, запятая - десятичная
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')

    s = re.sub(r'[^0-9\.-]', '', s)
    if s in {'', '-', '.', '-.'}:
        return np.nan

    try:
        return float(s)
    except Exception:
        return np.nan


def month_start(value):
    dt = pd.to_datetime(value, errors='coerce')
    if pd.isna(dt):
        return pd.NaT
    return pd.Timestamp(dt.year, dt.month, 1)

In [ ]:
# Загрузка и нормализация Excel
excel_frames = []
for src in excel_sources:
    report_month = month_start(src['report_month'])
    path = src['path']
    header = src.get('header', 0)

    df = pd.read_excel(path, header=header)

    col_inn = pick_column(df.columns, excel_inn_aliases + [excel_col_inn], 'ИНН', path)
    col_chod = pick_column(df.columns, excel_chod_aliases + [excel_col_chod], 'ЧОД', path)
    col_finrez = pick_column(df.columns, excel_finrez_aliases + [excel_col_finrez], 'Фин.Рез', path)
    col_term = pick_column(df.columns, excel_term_cnt_aliases + [excel_col_term_cnt], 'Количество терминалов', path)

    print(f"Файл: {path}")
    print(f"  header={header}")
    print(f"  ИНН -> {col_inn}")
    print(f"  ЧОД -> {col_chod}")
    print(f"  Фин.Рез -> {col_finrez}")
    print(f"  Терминалы -> {col_term}")

    cur = pd.DataFrame({
        'report_month': report_month,
        'inn': df[col_inn].apply(normalize_inn),
        'chod': df[col_chod].apply(parse_numeric),
        'fin_result': df[col_finrez].apply(parse_numeric),
        'term_cnt': df[col_term].apply(parse_numeric),
    })
    cur['source_file'] = path
    excel_frames.append(cur)

excel_raw = pd.concat(excel_frames, ignore_index=True)
excel_raw = excel_raw[excel_raw['report_month'].notna()]

excel_qc = (
    excel_raw.groupby('report_month', as_index=False)
    .agg(
        rows=('inn', 'size'),
        inn_cnt=('inn', 'nunique'),
        chod_sum=('chod', 'sum'),
        finrez_sum=('fin_result', 'sum'),
        finrez_pos_sum=('fin_result', lambda s: s[s > 0].sum()),
        finrez_neg_sum=('fin_result', lambda s: s[s < 0].sum()),
        term_cnt_sum=('term_cnt', 'sum'),
    )
    .sort_values('report_month')
)

display(excel_qc)

In [ ]:
# Агрегация Excel до уровня месяц + ИНН
excel_by_inn = (
    excel_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        excel_chod=('chod', 'sum'),
        excel_finrez=('fin_result', 'sum'),
        excel_term_cnt=('term_cnt', 'sum'),
    )
)

excel_by_inn['excel_finrez_pos'] = np.where(excel_by_inn['excel_finrez'] > 0, excel_by_inn['excel_finrez'], 0.0)
excel_by_inn['excel_finrez_neg'] = np.where(excel_by_inn['excel_finrez'] < 0, excel_by_inn['excel_finrez'], 0.0)

excel_by_month = (
    excel_by_inn.groupby('report_month', as_index=False)
    .agg(
        excel_rows=('inn', 'size'),
        excel_inn_cnt=('inn', 'nunique'),
        excel_chod_sum=('excel_chod', 'sum'),
        excel_finrez_sum=('excel_finrez', 'sum'),
        excel_finrez_pos_sum=('excel_finrez_pos', 'sum'),
        excel_finrez_neg_sum=('excel_finrez_neg', 'sum'),
        excel_term_cnt_sum=('excel_term_cnt', 'sum'),
    )
)

# Явный показатель количества клиентов
excel_by_month['excel_client_cnt'] = excel_by_month['excel_inn_cnt']

display(excel_by_month.sort_values('report_month'))

In [ ]:
# Загрузка данных из файлов озера
lake_frames = []
for src in lake_sources:
    report_month = month_start(src['report_month'])
    path = src['path']
    sep = src.get('sep', ',')

    df = pd.read_csv(path, sep=sep)

    col_inn = pick_column(df.columns, lake_inn_aliases + [lake_col_inn], 'inn', path)
    col_chod = pick_column(df.columns, lake_chod_aliases + [lake_col_chod], 'chod', path)
    col_finrez = pick_column(df.columns, lake_finrez_aliases + [lake_col_finrez], 'fin_result', path)
    col_term = pick_column(df.columns, lake_term_cnt_aliases + [lake_col_term_cnt], 'term_cnt', path)

    print(f"Файл озера: {path}")
    print(f"  inn -> {col_inn}")
    print(f"  chod -> {col_chod}")
    print(f"  fin_result -> {col_finrez}")
    print(f"  term_cnt -> {col_term}")

    cur = pd.DataFrame({
        'report_month': report_month,
        'inn': df[col_inn].apply(normalize_inn),
        'chod': df[col_chod].apply(parse_numeric),
        'fin_result': df[col_finrez].apply(parse_numeric),
        'term_cnt': df[col_term].apply(parse_numeric),
    })
    cur['source_file'] = path
    lake_frames.append(cur)

lake_raw = pd.concat(lake_frames, ignore_index=True)
lake_raw = lake_raw[lake_raw['report_month'].notna()]

lake_by_inn = (
    lake_raw.groupby(['report_month', 'inn'], as_index=False)
    .agg(
        lake_chod=('chod', 'sum'),
        lake_finrez=('fin_result', 'sum'),
        lake_term_cnt=('term_cnt', 'sum'),
    )
)

lake_by_inn['lake_finrez_pos'] = np.where(lake_by_inn['lake_finrez'] > 0, lake_by_inn['lake_finrez'], 0.0)
lake_by_inn['lake_finrez_neg'] = np.where(lake_by_inn['lake_finrez'] < 0, lake_by_inn['lake_finrez'], 0.0)

lake_by_month = (
    lake_by_inn.groupby('report_month', as_index=False)
    .agg(
        lake_rows=('inn', 'size'),
        lake_inn_cnt=('inn', 'nunique'),
        lake_chod_sum=('lake_chod', 'sum'),
        lake_finrez_sum=('lake_finrez', 'sum'),
        lake_finrez_pos_sum=('lake_finrez_pos', 'sum'),
        lake_finrez_neg_sum=('lake_finrez_neg', 'sum'),
        lake_term_cnt_sum=('lake_term_cnt', 'sum'),
    )
)

# Явный показатель количества клиентов
lake_by_month['lake_client_cnt'] = lake_by_month['lake_inn_cnt']

display(lake_by_month.sort_values('report_month'))

In [ ]:
# Таблица A: сравнение агрегатов по месяцам (Excel vs lake)
summary_a = (
    excel_by_month.merge(lake_by_month, on='report_month', how='outer')
    .sort_values('report_month')
    .reset_index(drop=True)
)

for col in [
    'excel_rows', 'excel_inn_cnt', 'excel_client_cnt', 'lake_rows', 'lake_inn_cnt', 'lake_client_cnt',
    'excel_chod_sum', 'excel_finrez_sum', 'excel_finrez_pos_sum', 'excel_finrez_neg_sum', 'excel_term_cnt_sum',
    'lake_chod_sum', 'lake_finrez_sum', 'lake_finrez_pos_sum', 'lake_finrez_neg_sum', 'lake_term_cnt_sum'
]:
    if col in summary_a.columns:
        summary_a[col] = summary_a[col].fillna(0)

summary_a['diff_client_cnt_abs'] = summary_a['lake_client_cnt'] - summary_a['excel_client_cnt']
summary_a['diff_chod_abs'] = summary_a['lake_chod_sum'] - summary_a['excel_chod_sum']
summary_a['diff_finrez_abs'] = summary_a['lake_finrez_sum'] - summary_a['excel_finrez_sum']
summary_a['diff_finrez_pos_abs'] = summary_a['lake_finrez_pos_sum'] - summary_a['excel_finrez_pos_sum']
summary_a['diff_finrez_neg_abs'] = summary_a['lake_finrez_neg_sum'] - summary_a['excel_finrez_neg_sum']
summary_a['diff_term_cnt_abs'] = summary_a['lake_term_cnt_sum'] - summary_a['excel_term_cnt_sum']

summary_a['diff_client_cnt_pct'] = np.where(
    summary_a['excel_client_cnt'] != 0,
    summary_a['diff_client_cnt_abs'] / summary_a['excel_client_cnt'] * 100,
    np.nan,
)
summary_a['diff_chod_pct'] = np.where(
    summary_a['excel_chod_sum'] != 0,
    summary_a['diff_chod_abs'] / summary_a['excel_chod_sum'] * 100,
    np.nan,
)
summary_a['diff_finrez_pct'] = np.where(
    summary_a['excel_finrez_sum'] != 0,
    summary_a['diff_finrez_abs'] / summary_a['excel_finrez_sum'] * 100,
    np.nan,
)
summary_a['diff_term_cnt_pct'] = np.where(
    summary_a['excel_term_cnt_sum'] != 0,
    summary_a['diff_term_cnt_abs'] / summary_a['excel_term_cnt_sum'] * 100,
    np.nan,
)

display(summary_a)

In [ ]:
# Таблица B: расхождения по ИНН (Excel vs lake)
comparison_b = (
    excel_by_inn.merge(
        lake_by_inn,
        on=['report_month', 'inn'],
        how='outer',
        indicator=True,
    )
)

for col in [
    'excel_chod', 'excel_finrez', 'excel_term_cnt',
    'lake_chod', 'lake_finrez', 'lake_term_cnt'
]:
    comparison_b[col] = comparison_b[col].fillna(0.0)

comparison_b['excel_finrez_pos'] = np.where(comparison_b['excel_finrez'] > 0, comparison_b['excel_finrez'], 0.0)
comparison_b['excel_finrez_neg'] = np.where(comparison_b['excel_finrez'] < 0, comparison_b['excel_finrez'], 0.0)
comparison_b['lake_finrez_pos'] = np.where(comparison_b['lake_finrez'] > 0, comparison_b['lake_finrez'], 0.0)
comparison_b['lake_finrez_neg'] = np.where(comparison_b['lake_finrez'] < 0, comparison_b['lake_finrez'], 0.0)

comparison_b['diff_chod'] = comparison_b['lake_chod'] - comparison_b['excel_chod']
comparison_b['diff_finrez'] = comparison_b['lake_finrez'] - comparison_b['excel_finrez']
comparison_b['diff_finrez_pos'] = comparison_b['lake_finrez_pos'] - comparison_b['excel_finrez_pos']
comparison_b['diff_finrez_neg'] = comparison_b['lake_finrez_neg'] - comparison_b['excel_finrez_neg']
comparison_b['diff_term_cnt'] = comparison_b['lake_term_cnt'] - comparison_b['excel_term_cnt']
comparison_b['total_abs_delta'] = (
    comparison_b['diff_chod'].abs()
    + comparison_b['diff_finrez'].abs()
    + comparison_b['diff_term_cnt'].abs()
)

comparison_b['reason_flag'] = np.select(
    [
        comparison_b['_merge'] == 'left_only',
        comparison_b['_merge'] == 'right_only',
        (comparison_b['_merge'] == 'both') & (
            (comparison_b['diff_chod'].abs() > 0.01)
            | (comparison_b['diff_finrez'].abs() > 0.01)
            | (comparison_b['diff_term_cnt'].abs() > 0.01)
        ),
    ],
    ['only_excel', 'only_lake', 'value_mismatch'],
    default='match',
)

comparison_b = comparison_b.drop(columns=['_merge'])
comparison_b = comparison_b.sort_values(['report_month', 'reason_flag', 'total_abs_delta'], ascending=[True, True, False])

print('Первые 50 расхождений (без полностью совпавших записей):')
display(comparison_b[comparison_b['reason_flag'] != 'match'].head(50))

In [ ]:
# Диагностика причин расхождений
reason_share = (
    comparison_b[comparison_b['reason_flag'] != 'match']
    .groupby(['report_month', 'reason_flag'], as_index=False)
    .agg(
        inn_cnt=('inn', 'nunique'),
        rows=('inn', 'size'),
        chod_abs_delta=('diff_chod', lambda s: s.abs().sum()),
        finrez_abs_delta=('diff_finrez', lambda s: s.abs().sum()),
        term_cnt_abs_delta=('diff_term_cnt', lambda s: s.abs().sum()),
        total_abs_delta=('total_abs_delta', 'sum'),
    )
    .sort_values(['report_month', 'reason_flag'])
)

# Доля вклада каждой причины в суммарное абсолютное расхождение за месяц
reason_share['month_total_abs_delta'] = reason_share.groupby('report_month')['total_abs_delta'].transform('sum')
reason_share['delta_share_pct'] = np.where(
    reason_share['month_total_abs_delta'] != 0,
    reason_share['total_abs_delta'] / reason_share['month_total_abs_delta'] * 100,
    np.nan,
)

display(reason_share)

# Эффект дублирования: строки vs уникальные ИНН
dup_effect = pd.concat(
    [
        excel_raw.groupby('report_month', as_index=False).agg(source=('report_month', lambda _: 'excel'), rows=('inn', 'size'), inn_cnt=('inn', 'nunique')),
        lake_raw.groupby('report_month', as_index=False).agg(source=('report_month', lambda _: 'lake'), rows=('inn', 'size'), inn_cnt=('inn', 'nunique')),
    ],
    ignore_index=True,
)

dup_effect['rows_per_inn'] = np.where(dup_effect['inn_cnt'] != 0, dup_effect['rows'] / dup_effect['inn_cnt'], np.nan)

display(dup_effect.sort_values(['report_month', 'source']))

print('TOP-20 ИНН по |delta_chod| + |delta_finrez| + |delta_term_cnt|:')
top_delta = comparison_b.sort_values('total_abs_delta', ascending=False)
display(top_delta.head(20))

In [ ]:
# Опционально: сохранить результаты сверки в CSV
save_outputs = True
output_prefix = 'chod_finrez_q1_reconciliation'

if save_outputs:
    summary_a.to_csv(f'{output_prefix}_summary_by_month.csv', index=False, encoding='utf-8-sig')
    comparison_b.to_csv(f'{output_prefix}_diff_by_inn.csv', index=False, encoding='utf-8-sig')
    reason_share.to_csv(f'{output_prefix}_reason_share.csv', index=False, encoding='utf-8-sig')
    print('Файлы сохранены:')
    print(f'- {output_prefix}_summary_by_month.csv')
    print(f'- {output_prefix}_diff_by_inn.csv')
    print(f'- {output_prefix}_reason_share.csv')